# Ubicación de sitios de lanzamiento con Folium
Marcadores, clusters de éxito/fallo y análisis de proximidad (costa, ciudad, ferrocarril, carretera).

In [ ]:
import folium, pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin,cos,sqrt,atan2,radians
spacex_df=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/spacex_launch_geo.csv')
spacex_df=spacex_df[['Launch Site','Lat','Long','class']]
launch_sites_df=spacex_df.groupby(['Launch Site'],as_index=False).first()[['Launch Site','Lat','Long']]
launch_sites_df

In [ ]:
nasa=[29.559684888503615,-95.0830971930759]
site_map=folium.Map(location=nasa,zoom_start=5)
for _,r in launch_sites_df.iterrows():
    coord=[r['Lat'],r['Long']]
    folium.Circle(coord,radius=1000,color='#d35400',fill=True).add_child(folium.Popup(r['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord,icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0),html='<div style="font-size:12;color:#d35400;"><b>%s</b></div>'%r['Launch Site'])).add_to(site_map)
site_map

## Marcar éxitos y fallos (verde / rojo)

In [ ]:
spacex_df['marker_color']=spacex_df['class'].apply(lambda c:'green' if c==1 else 'red')
marker_cluster=MarkerCluster();site_map.add_child(marker_cluster)
for _,r in spacex_df.iterrows():
    marker_cluster.add_child(folium.Marker([r['Lat'],r['Long']],icon=folium.Icon(color='white',icon_color=r['marker_color'])))
site_map

## Distancias a puntos cercanos

In [ ]:
site_map.add_child(MousePosition(position='topright',separator=' Long: ',empty_string='NaN',lng_first=False,num_digits=20,prefix='Lat:'))
def calculate_distance(lat1,lon1,lat2,lon2):
    R=6373.0
    la1,lo1,la2,lo2=map(radians,[lat1,lon1,lat2,lon2])
    a=sin((la2-la1)/2)**2+cos(la1)*cos(la2)*sin((lo2-lo1)/2)**2
    return R*2*atan2(sqrt(a),sqrt(1-a))
# Sitio KSC LC-39A y coordenadas aproximadas de la costa más cercana (ajústalas con MousePosition)
launch=(28.57325,-80.64608);coast=(28.56321,-80.56801)
d=calculate_distance(*launch,*coast)
folium.Marker(coast,icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0),html='<div style="font-size:12;color:#252526;"><b>%.2f KM</b></div>'%d)).add_to(site_map)
folium.PolyLine([launch,coast],weight=1).add_to(site_map)
print('Distancia a la costa: %.2f km'%d)
site_map

Repite el cálculo con las coordenadas de la ciudad, ferrocarril y carretera más cercanas (obtenidas con `MousePosition`) y dibuja una `PolyLine` para cada una.